In [1]:
import polars as pl

from ohlc_dss_model.data import load_parquet, remove_incomplete_days, intraday_session_tagging, session_tagging, filter_valid_sessions

from ohlc_dss_model.features import (
    aggregate_sessions, yang_zhang, 
    PRE_NY_SPEC, FULL_DAY_SPEC, 
    calculate_excursion_bands, assign_direction, 
    detect_pivots, pivot_extraction
)

from ohlc_dss_model.utils import convert_to_timezone, plot_session

from ohlc_dss_model.config import config

from pathlib import Path

from datetime import date

In [2]:
def load_raw_data(file_path: Path) -> pl.DataFrame:
    raw_data = load_parquet(file_path)
    raw_data = convert_to_timezone(raw_data)
    raw_data = session_tagging(raw_data)
    raw_data = intraday_session_tagging(raw_data)
    raw_data = remove_incomplete_days(raw_data)
    return raw_data.select(pl.col([
        "DateTime", "Session", "Intraday_Session", 
        "Open", "High", "Low", "Close", "Volume"
    ]))

In [3]:
raw_30min_data = load_raw_data(config.data.raw_folder_path / "nq_30m.parquet")
raw_1min_data = load_raw_data(config.data.raw_folder_path / "nq_1m.parquet")

aggregated_data = aggregate_sessions(raw_30min_data)
aggregated_data = filter_valid_sessions(aggregated_data)

aggregated_data = aggregated_data.with_columns(
    pl.col("O_Asia").alias("O_Ref")
)

aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

aggregated_data = assign_direction(aggregated_data)
aggregated_data = calculate_excursion_bands(aggregated_data)

In [4]:
print(raw_30min_data.head(3))

shape: (3, 8)
┌──────────────────┬────────────┬──────────────────┬─────────┬─────────┬─────────┬────────┬────────┐
│ DateTime         ┆ Session    ┆ Intraday_Session ┆ Open    ┆ High    ┆ Low     ┆ Close  ┆ Volume │
│ ---              ┆ ---        ┆ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---    │
│ datetime[ns, Ame ┆ date       ┆ str              ┆ f64     ┆ f64     ┆ f64     ┆ f64    ┆ u64    │
│ rica/New_York]   ┆            ┆                  ┆         ┆         ┆         ┆        ┆        │
╞══════════════════╪════════════╪══════════════════╪═════════╪═════════╪═════════╪════════╪════════╡
│ 2011-01-02       ┆ 2011-01-03 ┆ Asia             ┆ 2220.0  ┆ 2223.75 ┆ 2219.25 ┆ 2220.5 ┆ 889    │
│ 18:00:00 EST     ┆            ┆                  ┆         ┆         ┆         ┆        ┆        │
│ 2011-01-02       ┆ 2011-01-03 ┆ Asia             ┆ 2220.0  ┆ 2222.0  ┆ 2219.75 ┆ 2221.5 ┆ 238    │
│ 18:30:00 EST     ┆            ┆                  ┆         ┆         ┆     

In [8]:
print(raw_1min_data.head(3))

shape: (3, 8)
┌──────────────────┬────────────┬──────────────────┬─────────┬─────────┬────────┬─────────┬────────┐
│ DateTime         ┆ Session    ┆ Intraday_Session ┆ Open    ┆ High    ┆ Low    ┆ Close   ┆ Volume │
│ ---              ┆ ---        ┆ ---              ┆ ---     ┆ ---     ┆ ---    ┆ ---     ┆ ---    │
│ datetime[ns, Ame ┆ date       ┆ str              ┆ f64     ┆ f64     ┆ f64    ┆ f64     ┆ u64    │
│ rica/New_York]   ┆            ┆                  ┆         ┆         ┆        ┆         ┆        │
╞══════════════════╪════════════╪══════════════════╪═════════╪═════════╪════════╪═════════╪════════╡
│ 2011-01-02       ┆ 2011-01-03 ┆ Asia             ┆ 2220.0  ┆ 2223.0  ┆ 2220.0 ┆ 2222.75 ┆ 169    │
│ 18:00:00 EST     ┆            ┆                  ┆         ┆         ┆        ┆         ┆        │
│ 2011-01-02       ┆ 2011-01-03 ┆ Asia             ┆ 2222.0  ┆ 2222.75 ┆ 2222.0 ┆ 2222.25 ┆ 10     │
│ 18:01:00 EST     ┆            ┆                  ┆         ┆         ┆     

In [6]:
print(aggregated_data.drop_nulls().head(5))

shape: (5, 33)
┌────────────┬────────────┬──────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ Session    ┆ O_New York ┆ O_London ┆ O_Asia  ┆ … ┆ Band_FE_N ┆ Band_FE_N ┆ Band_FE_P ┆ Band_FE_P │
│ ---        ┆ ---        ┆ ---      ┆ ---     ┆   ┆ eg_Upper  ┆ eg_Lower  ┆ os_Upper  ┆ os_Lower  │
│ date       ┆ f64        ┆ f64      ┆ f64     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆          ┆         ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2011-01-24 ┆ 2270.0     ┆ 2274.0   ┆ 2268.75 ┆ … ┆ 2241.0046 ┆ 2236.9241 ┆ 2300.5758 ┆ 2296.4953 │
│            ┆            ┆          ┆         ┆   ┆ 77        ┆ 94        ┆ 06        ┆ 23        │
│ 2011-01-25 ┆ 2287.5     ┆ 2297.75  ┆ 2296.75 ┆ … ┆ 2267.7694 ┆ 2263.5156 ┆ 2329.9843 ┆ 2325.7305 │
│            ┆            ┆          ┆         ┆   ┆ 53        ┆ 6         ┆

In [7]:
print(aggregated_data.columns)

['Session', 'O_New York', 'O_London', 'O_Asia', 'H_New York', 'H_London', 'H_Asia', 'L_New York', 'L_London', 'L_Asia', 'C_New York', 'C_London', 'C_Asia', 'O_Ref', 'Sigma_Historical', 'Sigma_Today', 'Z_Body', 'Z_Sigma', 'Tau', 'Direction', '_delta_t', 'Band_AE_Pos_Center', 'Band_AE_Neg_Center', 'Band_FE_Pos_Center', 'Band_FE_Neg_Center', 'Band_AE_Neg_Upper', 'Band_AE_Neg_Lower', 'Band_AE_Pos_Upper', 'Band_AE_Pos_Lower', 'Band_FE_Neg_Upper', 'Band_FE_Neg_Lower', 'Band_FE_Pos_Upper', 'Band_FE_Pos_Lower']


***
### **XGBoost Volume-Based Tabular Multiasset Approach**